# XGBoost Incremental Classifier with Adaptive Rounds

Train XGBoost classifier with yearly incremental training using booster continuation.
Trees accumulate across years with adaptive boosting rounds via early stopping on PR-AUC.

## 1. Setup and Imports

In [2]:
import numpy as np
import pandas as pd
import xgboost as xgb
import zarr
import pickle
import json
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, classification_report
)
import warnings
warnings.filterwarnings('ignore')

print("Imports completed successfully")
print(f"XGBoost version: {xgb.__version__}")

Imports completed successfully
XGBoost version: 3.1.2


## 2. Configuration and Constants

In [4]:
# Paths
DATA_PATH = 'training_data_with_features.zarr'
SPLIT_PATH = 'data_split.npz'
MODELS_DIR = Path('models_xgb')
MODELS_DIR.mkdir(exist_ok=True)

# Training configuration
YEARS = [2017, 2018, 2019, 2020, 2021, 2022]  # Skip year 0 (2016)
YEAR_INDICES = list(range(1, 7))  # Indices 1-6
EARLY_STOPPING_ROUNDS = 50
NUM_BOOST_ROUND = 500
RANDOM_STATE = 42

# XGBoost parameters (will add scale_pos_weight after computing class weights)
XGB_PARAMS = {
    'objective': 'binary:logistic',
    'eval_metric': ['aucpr', 'auc'],  # Monitor both, early stop on first
    'tree_method': 'hist',
    'max_depth': 6,
    'eta': 0.1,  # Learning rate
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'seed': RANDOM_STATE,
    'verbosity': 0
}

# History container
training_history = {
    'year': [],
    'train_accuracy': [],
    'train_precision': [],
    'train_recall': [],
    'train_f1': [],
    'val_accuracy': [],
    'val_precision': [],
    'val_recall': [],
    'val_f1': [],
    'val_roc_auc': [],
    'val_pr_auc': [],
    'best_iteration': []  # Number of trees used
}

print(f"Models directory: {MODELS_DIR}")
print(f"Years to train: {YEARS}")
print(f"XGBoost params: {XGB_PARAMS}")

Models directory: models_xgb
Years to train: [2017, 2018, 2019, 2020, 2021, 2022]
XGBoost params: {'objective': 'binary:logistic', 'eval_metric': ['aucpr', 'auc'], 'tree_method': 'hist', 'max_depth': 6, 'eta': 0.1, 'subsample': 0.8, 'colsample_bytree': 0.8, 'seed': 42, 'verbosity': 0}


## 3. Load Data

In [5]:
# Load data splits
data_split = np.load(SPLIT_PATH, allow_pickle=True)
train_pixel_indices = data_split['train_pixel_indices']
val_pixel_indices = data_split['val_pixel_indices']
test_pixel_indices = data_split['test_pixel_indices']

print(f"Training pixels: {len(train_pixel_indices):,}")
print(f"Validation pixels: {len(val_pixel_indices):,}")
print(f"Test pixels: {len(test_pixel_indices):,}")

# Load training data with features
ds = zarr.open(DATA_PATH, mode='r')

print("\nDataset variables:")
print(list(ds.keys()))
print(f"\nDataset shape: {ds['s2_bands'].shape}")
print(f"Years dimension: {ds['year'][:]}")

Training pixels: 5,597,776
Validation pixels: 1,273,437
Test pixels: 1,283,992

Dataset variables:
['cube_idx', 'cube_name', 'dem', 'disturbances', 'nbr', 'nbr_delta', 'ndvi', 'ndvi_delta', 'ndwi', 'ndwi_delta', 'pixel', 's2_band', 's2_bands', 'x', 'y', 'year', 'year_disturbance']

Dataset shape: (8155205, 7, 7)
Years dimension: [2016 2017 2018 2019 2020 2021 2022]


## 4. Feature Preparation Function

Reuses the exact logic from SGD Classifier:
- Extract S2 bands (7), DEM (1), spectral indices (3), temporal deltas (3)
- Impute S2 NaNs with per-pixel temporal mean across all years
- Filter out label 255 (invalid)
- Drop remaining NaN samples
- Skip year_idx=0 entirely

In [6]:
def prepare_features_for_year(ds, pixel_indices, year_idx, scaler=None):
    """
    Prepare features for a specific year from training_data_with_features.zarr.
    
    Returns:
        X: Feature array (n_samples, n_features)
        y: Label array (n_samples,)
        scaler: Fitted scaler (if scaler=None) or the passed scaler
    """
    # Skip year 0 (2016)
    if year_idx == 0:
        print(f"Skipping year_idx={year_idx} (year 2016)")
        return np.array([]), np.array([]), scaler
    
    # Extract S2 bands (7 bands)
    s2_bands = ds['s2_bands'][pixel_indices, year_idx, :].astype(np.float64)
    
    # S2 imputation: fill NaNs with per-pixel temporal mean across all years
    s2_all_years = ds['s2_bands'][pixel_indices, :, :].astype(np.float64)
    s2_mean_per_pixel = np.nanmean(s2_all_years, axis=1)  # Shape: (n_pixels, 7)
    
    # Fill NaN values in current year with per-pixel mean
    for band_idx in range(s2_bands.shape[1]):
        nan_mask = np.isnan(s2_bands[:, band_idx])
        s2_bands[nan_mask, band_idx] = s2_mean_per_pixel[nan_mask, band_idx]
    
    # Extract DEM (1 feature)
    dem = ds['dem'][pixel_indices, year_idx:year_idx+1].astype(np.float64)
    
    # Extract spectral indices (3 features)
    ndvi = ds['ndvi'][pixel_indices, year_idx:year_idx+1].astype(np.float64)
    ndwi = ds['ndwi'][pixel_indices, year_idx:year_idx+1].astype(np.float64)
    nbr = ds['nbr'][pixel_indices, year_idx:year_idx+1].astype(np.float64)
    
    # Extract temporal deltas (3 features) - only available for year_idx >= 1
    if year_idx >= 1:
        # Delta indices are offset: delta[0] = year[1] - year[0]
        delta_idx = year_idx - 1
        ndvi_delta = ds['ndvi_delta'][pixel_indices, delta_idx:delta_idx+1].astype(np.float64)
        ndwi_delta = ds['ndwi_delta'][pixel_indices, delta_idx:delta_idx+1].astype(np.float64)
        nbr_delta = ds['nbr_delta'][pixel_indices, delta_idx:delta_idx+1].astype(np.float64)
    else:
        # For year 0, deltas are 0 (but we skip year 0 anyway)
        ndvi_delta = np.zeros((len(pixel_indices), 1))
        ndwi_delta = np.zeros((len(pixel_indices), 1))
        nbr_delta = np.zeros((len(pixel_indices), 1))
    
    # Concatenate all features (14 total: 7 S2 + 1 DEM + 3 indices + 3 deltas)
    X = np.hstack([s2_bands, dem, ndvi, ndwi, nbr, ndvi_delta, ndwi_delta, nbr_delta])
    
    # Extract labels
    y = ds['disturbances'][pixel_indices, year_idx].astype(np.int32)
    
    # Filter out invalid labels (255)
    valid_label_mask = np.isin(y, [0, 1])
    X = X[valid_label_mask]
    y = y[valid_label_mask]
    
    # Drop samples with any remaining NaN
    nan_mask = ~np.isnan(X).any(axis=1)
    X = X[nan_mask]
    y = y[nan_mask]
    
    # Scaling
    if scaler is None:
        # Fit scaler on first year training data
        scaler = StandardScaler()
        X = scaler.fit_transform(X)
        print(f"  Scaler fitted on year_idx={year_idx} training data")
    else:
        # Use existing scaler
        X = scaler.transform(X)
    
    return X, y, scaler

print("Feature preparation function defined")

Feature preparation function defined


## 5. Compute Class Weights

Collect all training labels across years 1-6 to compute balanced class weights once.

In [7]:
print("Computing class weights from all training years...")
all_train_labels = []

for year_idx in YEAR_INDICES:
    year_val = YEARS[YEAR_INDICES.index(year_idx)]
    print(f"  Collecting labels from year {year_val} (idx={year_idx})...")
    
    # Get labels without scaling (we don't need features yet)
    _, y_batch, _ = prepare_features_for_year(ds, train_pixel_indices, year_idx, scaler=None)
    all_train_labels.extend(y_batch)
    
    print(f"    Samples: {len(y_batch):,}, Positive: {y_batch.sum():,} ({y_batch.mean()*100:.2f}%)")

all_train_labels = np.array(all_train_labels)
print(f"\nTotal training samples: {len(all_train_labels):,}")
print(f"Class distribution: {np.bincount(all_train_labels)}")

# Compute balanced class weights
classes = np.array([0, 1])
class_weights_array = compute_class_weight('balanced', classes=classes, y=all_train_labels)
class_weight_dict = {classes[i]: class_weights_array[i] for i in range(len(classes))}

print(f"\nClass weights: {class_weight_dict}")

# Compute scale_pos_weight for XGBoost (ratio of negative to positive)
n_neg = (all_train_labels == 0).sum()
n_pos = (all_train_labels == 1).sum()
scale_pos_weight = n_neg / n_pos

print(f"scale_pos_weight for XGBoost: {scale_pos_weight:.4f}")

# Update XGBoost params with scale_pos_weight
XGB_PARAMS['scale_pos_weight'] = scale_pos_weight

# Clean up to free memory
del all_train_labels

Computing class weights from all training years...
  Scaler fitted on year_idx=1 training data
    Samples: 5,588,210, Positive: 103,050 (1.84%)
  Scaler fitted on year_idx=2 training data
    Samples: 5,588,498, Positive: 114,767 (2.05%)
  Scaler fitted on year_idx=3 training data
    Samples: 5,588,366, Positive: 113,169 (2.03%)
  Scaler fitted on year_idx=4 training data
    Samples: 5,589,205, Positive: 104,090 (1.86%)
  Scaler fitted on year_idx=5 training data
    Samples: 5,588,534, Positive: 85,050 (1.52%)
  Scaler fitted on year_idx=6 training data
    Samples: 5,592,392, Positive: 171,390 (3.06%)

Total training samples: 33,535,205
Class distribution: [32843689   691516]

Class weights: {0: 0.5105273801612237, 1: 24.247598754041846}
scale_pos_weight for XGBoost: 47.4952


## 6. Yearly Incremental Training Loop

Train XGBoost incrementally:
- Year 1: Train new booster from scratch
- Years 2-6: Continue training with `xgb_model=prev_booster` to append trees
- Use early stopping on validation PR-AUC with adaptive rounds

In [8]:
print("="*80)
print("STARTING YEARLY INCREMENTAL TRAINING")
print("="*80)

scaler = None
prev_booster = None

for year_idx in YEAR_INDICES:
    year_val = YEARS[YEAR_INDICES.index(year_idx)]
    
    print(f"\n{'='*80}")
    print(f"YEAR {year_val} (index {year_idx})")
    print(f"{'='*80}")
    
    # Prepare data for this year
    print("\nPreparing training data...")
    X_train, y_train, scaler = prepare_features_for_year(ds, train_pixel_indices, year_idx, scaler)
    print(f"  Training samples: {len(X_train):,}, Features: {X_train.shape[1]}")
    print(f"  Class distribution: 0={np.sum(y_train==0):,}, 1={np.sum(y_train==1):,}")
    
    print("\nPreparing validation data...")
    X_val, y_val, _ = prepare_features_for_year(ds, val_pixel_indices, year_idx, scaler)
    print(f"  Validation samples: {len(X_val):,}")
    print(f"  Class distribution: 0={np.sum(y_val==0):,}, 1={np.sum(y_val==1):,}")
    
    # Create sample weights for training set
    sample_weights_train = np.array([class_weight_dict[int(label)] for label in y_train])
    
    # Create DMatrices
    dtrain = xgb.DMatrix(X_train, label=y_train, weight=sample_weights_train)
    dval = xgb.DMatrix(X_val, label=y_val)
    
    # Train XGBoost with incremental learning
    print("\nTraining XGBoost...")
    evals = [(dtrain, 'train'), (dval, 'val')]
    
    if prev_booster is None:
        # Year 1: Train from scratch
        print("  Training new booster from scratch")
        booster = xgb.train(
            XGB_PARAMS,
            dtrain,
            num_boost_round=NUM_BOOST_ROUND,
            evals=evals,
            early_stopping_rounds=EARLY_STOPPING_ROUNDS,
            verbose_eval=False
        )
    else:
        # Years 2-6: Continue training (append trees)
        print(f"  Continuing training from previous booster ({prev_booster.num_boosted_rounds()} existing trees)")
        booster = xgb.train(
            XGB_PARAMS,
            dtrain,
            num_boost_round=NUM_BOOST_ROUND,
            evals=evals,
            early_stopping_rounds=EARLY_STOPPING_ROUNDS,
            verbose_eval=False,
            xgb_model=prev_booster
        )
    
    best_iteration = booster.best_iteration
    total_trees = booster.num_boosted_rounds()
    print(f"  Training completed: {total_trees} total trees (best iteration: {best_iteration})")
    
    # Make predictions for metrics
    y_train_pred_proba = booster.predict(dtrain)
    y_train_pred = (y_train_pred_proba >= 0.5).astype(int)
    
    y_val_pred_proba = booster.predict(dval)
    y_val_pred = (y_val_pred_proba >= 0.5).astype(int)
    
    # Compute training metrics
    train_accuracy = accuracy_score(y_train, y_train_pred)
    train_precision = precision_score(y_train, y_train_pred, zero_division=0)
    train_recall = recall_score(y_train, y_train_pred, zero_division=0)
    train_f1 = f1_score(y_train, y_train_pred, zero_division=0)
    
    # Compute validation metrics
    val_accuracy = accuracy_score(y_val, y_val_pred)
    val_precision = precision_score(y_val, y_val_pred, zero_division=0)
    val_recall = recall_score(y_val, y_val_pred, zero_division=0)
    val_f1 = f1_score(y_val, y_val_pred, zero_division=0)
    
    # Compute AUC metrics (only if both classes present)
    if len(np.unique(y_val)) > 1:
        val_roc_auc = roc_auc_score(y_val, y_val_pred_proba)
        val_pr_auc = average_precision_score(y_val, y_val_pred_proba)
    else:
        val_roc_auc = 0.0
        val_pr_auc = 0.0
    
    # Log metrics
    training_history['year'].append(year_val)
    training_history['train_accuracy'].append(train_accuracy)
    training_history['train_precision'].append(train_precision)
    training_history['train_recall'].append(train_recall)
    training_history['train_f1'].append(train_f1)
    training_history['val_accuracy'].append(val_accuracy)
    training_history['val_precision'].append(val_precision)
    training_history['val_recall'].append(val_recall)
    training_history['val_f1'].append(val_f1)
    training_history['val_roc_auc'].append(val_roc_auc)
    training_history['val_pr_auc'].append(val_pr_auc)
    training_history['best_iteration'].append(best_iteration)
    
    # Print metrics
    print(f"\n  Training Metrics:")
    print(f"    Accuracy:  {train_accuracy:.4f}")
    print(f"    Precision: {train_precision:.4f}")
    print(f"    Recall:    {train_recall:.4f}")
    print(f"    F1:        {train_f1:.4f}")
    
    print(f"\n  Validation Metrics:")
    print(f"    Accuracy:  {val_accuracy:.4f}")
    print(f"    Precision: {val_precision:.4f}")
    print(f"    Recall:    {val_recall:.4f}")
    print(f"    F1:        {val_f1:.4f}")
    print(f"    ROC-AUC:   {val_roc_auc:.4f}")
    print(f"    PR-AUC:    {val_pr_auc:.4f}")
    
    # Save year-specific model
    year_model_path = MODELS_DIR / f'model_year_{year_val}.json'
    booster.save_model(year_model_path)
    print(f"\n  Model saved to {year_model_path}")
    
    # Save scaler after first year
    if prev_booster is None:
        scaler_path = MODELS_DIR / 'xgboost_scaler.pkl'
        with open(scaler_path, 'wb') as f:
            pickle.dump(scaler, f)
        print(f"  Scaler saved to {scaler_path}")
    
    # Update prev_booster for next iteration
    prev_booster = booster
    
    # Clean up to free memory
    del X_train, y_train, X_val, y_val, dtrain, dval

print(f"\n{'='*80}")
print("TRAINING COMPLETED")
print(f"{'='*80}")

STARTING YEARLY INCREMENTAL TRAINING

YEAR 2017 (index 1)

Preparing training data...
  Scaler fitted on year_idx=1 training data
  Training samples: 5,588,210, Features: 14
  Class distribution: 0=5,485,160, 1=103,050

Preparing validation data...
  Validation samples: 1,271,997
  Class distribution: 0=1,248,082, 1=23,915

Training XGBoost...
  Training new booster from scratch
  Training completed: 61 total trees (best iteration: 10)

  Training Metrics:
    Accuracy:  0.0596
    Precision: 0.0192
    Recall:    1.0000
    F1:        0.0377

  Validation Metrics:
    Accuracy:  0.0681
    Precision: 0.0194
    Recall:    0.9816
    F1:        0.0381
    ROC-AUC:   0.8004
    PR-AUC:    0.1187

  Model saved to models_xgb\model_year_2017.json
  Scaler saved to models_xgb\xgboost_scaler.pkl

YEAR 2018 (index 2)

Preparing training data...
  Training samples: 5,588,498, Features: 14
  Class distribution: 0=5,473,731, 1=114,767

Preparing validation data...
  Validation samples: 1,272,00

## 7. Save Training History

In [9]:
# Convert to DataFrame and save
history_df = pd.DataFrame(training_history)
history_path = 'xgb_history.csv'
history_df.to_csv(history_path, index=False)

print(f"Training history saved to {history_path}")
print("\nTraining History:")
print(history_df.to_string(index=False))

Training history saved to xgb_history.csv

Training History:
 year  train_accuracy  train_precision  train_recall  train_f1  val_accuracy  val_precision  val_recall   val_f1  val_roc_auc  val_pr_auc  best_iteration
 2017        0.059566         0.019231      0.999961  0.037736      0.068137       0.019427    0.981602 0.038100     0.800364    0.118705              10
 2018        0.261168         0.027044      0.999991  0.052663      0.269971       0.028693    0.973573 0.055742     0.881587    0.359543             500
 2019        0.106329         0.022017      0.993355  0.043080      0.113298       0.019947    0.987181 0.039104     0.667004    0.026146             552
 2020        0.164316         0.021701      0.995293  0.042476      0.176304       0.019838    0.985266 0.038892     0.721832    0.030250            1102
 2021        0.139751         0.017234      0.991123  0.033880      0.144956       0.017633    0.985457 0.034646     0.702736    0.025530            1103
 2022        0.

## 8. Final Evaluation on Combined Test Set

In [10]:
print("="*80)
print("FINAL EVALUATION ON COMBINED TEST SET")
print("="*80)

# Collect test data from all years
X_test_all = []
y_test_all = []

for year_idx in YEAR_INDICES:
    year_val = YEARS[YEAR_INDICES.index(year_idx)]
    print(f"\nLoading test data for year {year_val}...")
    
    X_test_year, y_test_year, _ = prepare_features_for_year(ds, test_pixel_indices, year_idx, scaler)
    X_test_all.append(X_test_year)
    y_test_all.append(y_test_year)
    
    print(f"  Samples: {len(X_test_year):,}")

# Concatenate all years
X_test = np.vstack(X_test_all)
y_test = np.concatenate(y_test_all)

print(f"\nTotal test samples: {len(X_test):,}")
print(f"Class distribution: 0={np.sum(y_test==0):,}, 1={np.sum(y_test==1):,}")

# Create DMatrix for test set
dtest = xgb.DMatrix(X_test, label=y_test)

# Make predictions with final booster
y_test_pred_proba = booster.predict(dtest)
y_test_pred = (y_test_pred_proba >= 0.5).astype(int)

# Compute metrics
test_accuracy = accuracy_score(y_test, y_test_pred)
test_precision = precision_score(y_test, y_test_pred, zero_division=0)
test_recall = recall_score(y_test, y_test_pred, zero_division=0)
test_f1 = f1_score(y_test, y_test_pred, zero_division=0)
test_roc_auc = roc_auc_score(y_test, y_test_pred_proba)
test_pr_auc = average_precision_score(y_test, y_test_pred_proba)
test_cm = confusion_matrix(y_test, y_test_pred)

print("\n" + "="*80)
print("FINAL TEST RESULTS")
print("="*80)
print(f"Accuracy:  {test_accuracy:.4f}")
print(f"Precision: {test_precision:.4f}")
print(f"Recall:    {test_recall:.4f}")
print(f"F1 Score:  {test_f1:.4f}")
print(f"ROC-AUC:   {test_roc_auc:.4f}")
print(f"PR-AUC:    {test_pr_auc:.4f}")
print(f"\nConfusion Matrix:")
print(test_cm)

# Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_test_pred, target_names=['No Disturbance', 'Disturbance']))

FINAL EVALUATION ON COMBINED TEST SET

Loading test data for year 2017...
  Samples: 1,281,605

Loading test data for year 2018...
  Samples: 1,281,667

Loading test data for year 2019...
  Samples: 1,281,583

Loading test data for year 2020...
  Samples: 1,281,805

Loading test data for year 2021...
  Samples: 1,281,594

Loading test data for year 2022...
  Samples: 1,282,451

Total test samples: 7,690,705
Class distribution: 0=7,518,869, 1=171,836

FINAL TEST RESULTS
Accuracy:  0.0884
Precision: 0.0238
Recall:    0.9938
F1 Score:  0.0465
ROC-AUC:   0.6385
PR-AUC:    0.0287

Confusion Matrix:
[[ 508916 7009953]
 [   1064  170772]]

Classification Report:
                precision    recall  f1-score   support

No Disturbance       1.00      0.07      0.13   7518869
   Disturbance       0.02      0.99      0.05    171836

      accuracy                           0.09   7690705
     macro avg       0.51      0.53      0.09   7690705
  weighted avg       0.98      0.09      0.12   769070

## 9. Save Final Results and Models

In [11]:
# Save test results to JSON
test_results = {
    'accuracy': float(test_accuracy),
    'precision': float(test_precision),
    'recall': float(test_recall),
    'f1': float(test_f1),
    'roc_auc': float(test_roc_auc),
    'pr_auc': float(test_pr_auc),
    'confusion_matrix': test_cm.tolist(),
    'total_samples': int(len(y_test)),
    'total_trees': int(booster.num_boosted_rounds())
}

test_results_path = 'xgboost_test_results.json'
with open(test_results_path, 'w') as f:
    json.dump(test_results, f, indent=2)

print(f"Test results saved to {test_results_path}")

# Save final model
final_model_path = MODELS_DIR / 'xgboost_model.json'
booster.save_model(final_model_path)
print(f"Final model saved to {final_model_path}")

# Scaler already saved during training
print(f"\nAll outputs saved to:")
print(f"  - Training history: xgb_history.csv")
print(f"  - Test results: xgboost_test_results.json")
print(f"  - Final model: {final_model_path}")
print(f"  - Scaler: {MODELS_DIR / 'xgboost_scaler.pkl'}")
print(f"  - Year models: {MODELS_DIR / 'model_year_YYYY.json'}")

Test results saved to xgboost_test_results.json
Final model saved to models_xgb\xgboost_model.json

All outputs saved to:
  - Training history: xgb_history.csv
  - Test results: xgboost_test_results.json
  - Final model: models_xgb\xgboost_model.json
  - Scaler: models_xgb\xgboost_scaler.pkl
  - Year models: models_xgb\model_year_YYYY.json


## 10. Summary Statistics

In [12]:
print("="*80)
print("TRAINING SUMMARY")
print("="*80)

print(f"\nTotal years trained: {len(YEARS)}")
print(f"Final total trees: {booster.num_boosted_rounds()}")
print(f"\nTrees added per year:")
for i, year in enumerate(YEARS):
    best_iter = training_history['best_iteration'][i]
    if i == 0:
        trees_added = best_iter
    else:
        trees_added = best_iter
    print(f"  {year}: ~{trees_added} trees (stopped at iteration {best_iter})")

print(f"\nValidation PR-AUC progression:")
for year, pr_auc in zip(training_history['year'], training_history['val_pr_auc']):
    print(f"  {year}: {pr_auc:.4f}")

print(f"\nFinal Test Performance:")
print(f"  Accuracy:  {test_accuracy:.4f}")
print(f"  Precision: {test_precision:.4f}")
print(f"  Recall:    {test_recall:.4f}")
print(f"  F1:        {test_f1:.4f}")
print(f"  ROC-AUC:   {test_roc_auc:.4f}")
print(f"  PR-AUC:    {test_pr_auc:.4f}")

print(f"\n{'='*80}")
print("NOTEBOOK COMPLETED SUCCESSFULLY")
print("="*80)

TRAINING SUMMARY

Total years trained: 6
Final total trees: 1205

Trees added per year:
  2017: ~10 trees (stopped at iteration 10)
  2018: ~500 trees (stopped at iteration 500)
  2019: ~552 trees (stopped at iteration 552)
  2020: ~1102 trees (stopped at iteration 1102)
  2021: ~1103 trees (stopped at iteration 1103)
  2022: ~1154 trees (stopped at iteration 1154)

Validation PR-AUC progression:
  2017: 0.1187
  2018: 0.3595
  2019: 0.0261
  2020: 0.0302
  2021: 0.0255
  2022: 0.0324

Final Test Performance:
  Accuracy:  0.0884
  Precision: 0.0238
  Recall:    0.9938
  F1:        0.0465
  ROC-AUC:   0.6385
  PR-AUC:    0.0287

NOTEBOOK COMPLETED SUCCESSFULLY
